# 🎯 Architecture: AWS Data Platform Serverless

**Advanced AWS Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** S3 Data Lakes, AWS Glue, Amazon Athena Serverless SQL
- **Tags:** `aws` | `s3` | `glue` | `athena` | `interview-prep`

## 📖 The Core Concept in Plain Language

AWS has three services that form the backbone of most modern data lakes. S3 acts as the massive, infinitely scalable hard drive. Glue acts as the catalog directory and ETL runner. Athena is a serverless SQL engine that reads files directly from S3 as if they were a database.

### Why This Matters

- **Serverless:** You manage zero servers. No Hadoop clusters to maintain.
- **Separation of Compute and Storage:** Keep petabytes of data on cheap S3; pay for compute ONLY when you run an Athena query.
- **Interoperability:** Because data is in open formats like Parquet, any tool (Spark, Python, Athena) can read it simultaneously.

### The Key Insight

```text
Raw Event Data → S3 (Data Lake in Parquet)
                      ↓
              AWS Glue Crawler (Discovers Schema)
                      ↓
              Glue Data Catalog (Metastore)
                      ↓
              Amazon Athena (Serverless SQL Query)
```

## 🔍 The S3 Foundation

In [ ]:
# -------------------------------------------------------
# S3 Partitioning is the #1 Performance Factor
# -------------------------------------------------------
print("S3 is an object store, not a file system. Keys are full string paths.")
print("Partitioning syntax: s3://bucket/data/year=2026/month=02/day=28/")
print("When Athena queries WHERE year='2026', it completely ignores all other folders.")

## 🏗️ The Anatomy of AWS Glue

Glue has two main parts: The **Data Catalog** (like a Hive metastore) and the **ETL Engine** (managed Serverless PySpark).

### Structure:
```python
# A Glue PySpark Job
import sys
from awsglue.context import GlueContext
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())

# Load from catalog table (no raw S3 paths needed!)
dynamic_frame = glueContext.create_dynamic_frame.from_catalog(
    database="telemetry_db", table_name="servers"
)
```

In [ ]:
print("Glue Crawlers: Automatically scan S3, infer CSV/JSON schemas, and build table definitions.")

## 🔄 Amazon Athena SQL

Athena uses Presto under the hood to execute SQL queries on flat files in S3.

In [ ]:
# -------------------------------------------------------
# Approach 1: Scanning standard CSV (Expensive & Slow)
# -------------------------------------------------------
print("Querying a 1TB CSV file scans all 1TB. You pay $5 per query. Slow.")

# -------------------------------------------------------
# Approach 2: Scanning Parquet with Partitions (Cheap & Fast)
# -------------------------------------------------------
print("Querying partitioned Parquet (Columnar): Athena only reads the columns requested, and only in the right date folders.")
print("A 1TB dataset query might only scan 10GB. You pay $0.05 per query. Blazing fast.")

## 🌍 Real-World Example: Serverless Telemetry Pipeline

**The Citi Context:** We needed to ingest daily metrics from thousands of endpoints without managing databases that would become bottlenecks. We used the S3 + Athena pattern to achieve infinite scale natively.

### Traditional Database (Bottleneck)
```text
6000 agents inserting SQL rows -> Database locking, storage limits, CPU alerts.
```

### With AWS Data Platform (Serverless)
```text
Kinesis Firehose -> dumps raw Parquet to S3 paths partitioned by day -> Athena queries it instantly.
```

In [ ]:
print("Impact: Storage costs dropped to pennies. Compute scales elastically.")
print("No database administrator needed.")

## 🎭 Advanced: Parquet Columnar Storage

In [ ]:
# Why Parquet matters for Athena
# -------------------------------------------------------
print("Row-based (CSV): Values stored sequentially per row. `SELECT avg_cpu FROM tbl` forces a scan of all data.")
print("Columnar (Parquet): All `avg_cpu` values stored contiguously. `SELECT avg_cpu` ignores all other column blocks.")

## 🎤 5 Interview Q&A

### Q1: What is a Glue Crawler?

**Answer:** A Glue Crawler is a managed job that connects to a data store (like an S3 bucket), reads the first few Megabytes of data, infers the schema (columns, data types, JSON structures), and creates or updates a table metadata definition in the Glue Data Catalog automatically.

---

### Q2: How does Athena pricing work and how do you optimize it?

**Answer:** Athena charges $5 per Terabyte *scanned* (not stored). You optimize it via 1) Compression (Snappy/GZIP), 2) Columnar formats (Parquet/ORC) so only queried columns are scanned, and 3) S3 Partitioning so entire folders of irrelevant data are pruned via `WHERE` clauses.

---

### Q3: What is the difference between AWS Glue and Amazon EMR?

**Answer:** Glue is a managed, serverless ETL service — you write the PySpark script and AWS spins up the containers invisibly. EMR is a managed Hadoop/Spark cluster — you have root access to the EC2 instances, you manage the nodes, cluster scaling, and configurations. Glue is easier; EMR is more flexible.

---

### Q4: Why use Lake Formation?

**Answer:** Lake Formation wraps around S3 and the Glue Catalog to provide fine-grained, database-style access control. Without it, you can only grant access to whole S3 buckets. With it, you can grant access to specific columns or rows for different IAM roles (essential for healthcare/finance).

---

### Q5: How do you handle "late-arriving" data in partitioned S3 buckets?

**Answer:** Partition by event timestamp, not ingestion timestamp. When late data arrives, it drops into the correct historical partition folder. We then re-run the daily aggregation script for that date, or use an architecture like Apache Hudi or Iceberg on top of S3 to handle upserts/updates.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Data Lake** | Centralized repository for structured and unstructured data at any scale | S3 bucket of Parquet files |
| **Partition Pruning** | Skipping data folders that don't match the query filter | `WHERE year='2026'` skips '2025' |
| **Serverless SQL** | Query engines without provisioning underlying database servers | Amazon Athena |
| **Columnar Format** | Storing data contiguously by column rather than by row | Apache Parquet, ORC |
| **Data Catalog** | Metastore holding database, table, and schema definitions | AWS Glue Data Catalog |
| **DynamicFrame** | Glue's extension of a Spark DataFrame that handles messy schema changes | `glueContext.create_dynamic_frame` |

## 💼 The Citi Narrative: Serverless Scale

### The Problem

Collecting telemetry from 6,000+ endpoints generated massive amounts of JSON data. Loading this daily log data into standard relational databases triggered high latency, index locking, and massive scaling costs. We needed a decoupled storage and compute layer.

### The Solution

I embraced the AWS Serverless Data Lake pattern. We persisted all incoming raw payloads directly to S3 as Parquet files partitioned by date. Analysts simply ran Amazon Athena queries to build their capacity reports directly on top of the S3 files.

### The Code Pattern

```sql
-- In Athena:
SELECT server_id, MAX(cpu) 
FROM telemetry 
WHERE year_month = '2026-02';
```

### The Impact

- **Storage Costs:** Dropped precipitously using S3 rather than attached block storage (EBS) for databases.
- **Query Speed:** Columnar Parquet + partitioned structures reduced querying times from hours to seconds.
- **Zero Maintenance:** No index fragmentation, no vacuums, no DB administration. It "just scaled".

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: S3 Partitioning Concept
# If you query `SELECT * FROM tbl WHERE state='NY'`, 
# how should your S3 path be structured to optimize this?

print("s3://bucket/data/state=NY/file.parquet")

In [ ]:
# EXERCISE 2: Glue vs EMR
# You have a small PySpark script that runs nightly. Do you launch EMR or Glue?

print("Glue. It's serverless, you only pay for the minutes it runs, no cluster setup overhead.")

In [ ]:
# EXERCISE 3: Format efficiencies
# Why not use JSON in S3 for Athena queries?

print("JSON is row-oriented. Athena must scan the entire file to find one field. Parquet is columnar.")

## 🎯 Summary: Key Takeaways

### What You've Learned

1. ✅ **S3:** The foundation. Infinitely scalable object store.
2. ✅ **Partitioning:** The key to saving money and speeding up queries via pruning.
3. ✅ **Glue Catalog:** Replaces Hive Metastore. Manages schemas.
4. ✅ **Glue Crawlers:** Ingest files and define tables automatically.
5. ✅ **Athena:** Serverless Trino/Presto SQL. Queries files directly.
6. ✅ **Parquet:** Columnar storage essential for performant Athena use.
7. ✅ **Lake Formation:** How you apply row/column security to S3 objects.

### When to Use This Pattern

- ✅ **DO** use S3+Athena for ad-hoc analytical queries over historical log/event data.
- ✅ **DO** convert raw unstructured data to Parquet using Glue before querying.
- ✅ **DO** partition thoroughly by date or high-cardinality query keys.
- ❌ **DON'T** use Athena for real-time OLTP point lookups (latency is too high).
- ❌ **DON'T** run Athena directly against thousands of tiny 5KB JSON files. (Compact them into large Parquet files).

### Interview Confidence Checklist

- [ ] Predict how an Athena query executes over S3 partitions.
- [ ] Explain why Parquet saves money over CSV.
- [ ] Contrast Glue vs EMR.
- [ ] Outline the full architectural flow (Raw -> S3 -> Crawler -> Catalog -> Athena).
- [ ] Present the Citi telemetry story using these concepts.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Mastering this triad unlocks massive scale. 🚀